# Module 14 — Raw Agent Loop Engineering
Build an agent without a framework: Observe → Decide → Act → Verify → Stop/Recover.

In [ ]:
from dataclasses import dataclass,field
from enum import Enum
class Kind(Enum): FINAL='final'; TOOL='tool'; REFLECT='reflect'; ABORT='abort'
@dataclass
class Decision: kind:Kind; tool:str|None=None; args:dict=field(default_factory=dict); reason:str=''
@dataclass
class State: goal:str; observations:list=field(default_factory=list); results:list=field(default_factory=list); steps:int=0

In [ ]:
tools={'lookup':lambda x:x*2}
def decide(s):
    if s.results:return Decision(Kind.FINAL,reason='verified')
    return Decision(Kind.TOOL,tool='lookup',args={'x':21})
def run(s,max_steps=5):
    while s.steps<max_steps:
        d=decide(s); s.steps+=1
        if d.kind is Kind.FINAL:return s,'final'
        if d.kind is Kind.TOOL:
            if d.tool not in tools:return s,'invalid_tool'
            s.results.append(tools[d.tool](**d.args)); s.observations.append(s.results[-1])
    return s,'step_budget'
s,reason=run(State('find answer')); print(s.results,reason)

## Detailed exercises
1. Add a typed decision schema.
2. Add 3 tools.
3. Add tool authorization.
4. Add step/token/tool-call budgets.
5. Add repeated-state fingerprints.
6. Add oscillation detection.
7. Add verification before FINAL.
8. Add transient failure retry.
9. Add reflection with a bounded reflection budget.
10. Persist and replay trajectories.
11. Add cancellation/deadline handling.
12. Compare reactive and planner/executor agents on 100 tasks.

## Failure injection
Infinite loop; two-state oscillation; premature final answer; malicious tool result; retry storm; invalid tool; impossible goal.

For every failure: identify the runtime control that stops it, then write a regression test.

## Gold challenge
Build three loop architectures and benchmark task success, steps, tool calls, latency, cost, verification failures and unsafe-action attempts. Defend the minimum sufficient design.